# 02 — Extract MorphAgent features from images

`source/feature_library/` holds one folder per feature:

- **439 code features** — `extract.py` with `extract(img, seg) → float`
- **28 VLM features** — a planner record scored by a vision model (0–100)

Images come from `data/dataset/` (notebook 01). This notebook runs a **smoke test** on a few samples. A full 3,552-image × 467-feature pass is documented but not launched here.


In [2]:
from pathlib import Path
import sys

HERE = Path.cwd().resolve()
if HERE.name == "notebook":
    TUTORIAL = HERE.parent
elif (HERE / "code").is_dir() and (HERE / "source").is_dir():
    TUTORIAL = HERE
else:
    TUTORIAL = HERE
sys.path.insert(0, str(TUTORIAL / "code"))

import importlib
import paths as P
P = importlib.reload(P)
print("Tutorial root:", P.ROOT)
print("data/        :", P.DATA_DIR)
print("source/      :", P.SOURCE_DIR)
print("code/        :", P.CODE_DIR)

import pandas as pd
from IPython.display import display
import extract_morphagent_features as feat
feat = importlib.reload(feat)

manifest = pd.read_csv(P.FEATURE_LIBRARY / "manifest.csv")
display(manifest["method"].value_counts().to_frame("n"))
print("Library:", P.FEATURE_LIBRARY)
print("Dataset:", P.DATASET_DIR)


Tutorial root: /Users/yez/Desktop/tutorial_BBBC021
data/        : /Users/yez/Desktop/tutorial_BBBC021/data
source/      : /Users/yez/Desktop/tutorial_BBBC021/source
code/        : /Users/yez/Desktop/tutorial_BBBC021/code


,n
method,
code,439
vlm,28


Library: /Users/yez/Desktop/tutorial_BBBC021/source/feature_library
Dataset: /Users/yez/Desktop/tutorial_BBBC021/data/dataset


## 1. Code feature smoke

No API is required. The extractor loads `image.tif` plus `segmentation/*.tif`.


In [3]:
from IPython.display import display

ids = feat.list_sample_ids(feat.require_dataset())
print("n samples:", len(ids))
print("first 5 :", ids[:5])

code_feature = "tubulin_intensity_total"
code_path = P.FEATURE_LIBRARY / "code" / code_feature / "extract.py"
print("\n---", code_path, "---")
print(code_path.read_text(encoding="utf-8")[:1600])

code_df = feat.run_code_feature(code_feature, P.DATASET_DIR, ids[:5])
feat.OUTPUT_SMOKE.mkdir(parents=True, exist_ok=True)
code_df.to_csv(feat.OUTPUT_SMOKE / f"smoke_code_{code_feature}.csv", index=False)
display(code_df)


n samples: 3552
first 5 : ['10064_alsterpaullone', '10065_alsterpaullone', '10066_alsterpaullone', '10067_alsterpaullone', '10068_alsterpaullone']

--- /Users/yez/Desktop/tutorial_BBBC021/source/feature_library/code/tubulin_intensity_total/extract.py ---
def extract(img, *segmentation_masks):
    # IMPORT ALL REQUIRED PACKAGES AT THE BEGINNING OF THE FUNCTION
    import numpy as np
    from scipy import ndimage

    # 1. Input Validation and Preparation
    # Ensure image is a numpy array
    img = np.asarray(img)
    
    # Check dimensionality. We expect (H, W, C) = (512, 512, 3) or similar 2D multichannel
    if img.ndim != 3 or img.shape[2] < 2:
        # If dimensions are unexpected (e.g., 2D grayscale), return 0.0
        return 0.0

    # 2. Extract and Normalize the Tubulin Channel
    # According to dataset info: Channel 1 = Green = Tubulin
    # Convert to float64 for precision during summation and normalize to [0, 1]
    tubulin_channel = img[:, :, 1].astype(np.float64) / 25

,sample_id,tubulin_intensity_total,error
0,10064_alsterpaullone,25768.294118,None
1,10065_alsterpaullone,16957.847059,None
2,10066_alsterpaullone,24360.988235,None
3,10067_alsterpaullone,16144.545098,None
4,10068_alsterpaullone,13966.960784,None


## 2. VLM credentials (your endpoint)

This tutorial **does not ship** a base URL or API key. Fill in an OpenAI-compatible vision endpoint, or leave the strings empty to skip the live VLM call.

Do not use `input()` / `getpass()` here — paste into the variables below and re-run the cell.


In [4]:
# Your OpenAI-compatible vision API (leave blank to skip VLM).
VLM_API_BASE_URL = "https://api.gpugeek.com/v1"   # e.g. "https://api.example.com/v1"
VLM_API_KEY = "00uqb6eym2n5od01000dkbm4s5ez13ok00p51kr1"
VLM_MODEL = "gpt-5.5"

if VLM_API_BASE_URL.strip() and VLM_API_KEY.strip():
    feat.configure_vlm_api(VLM_API_BASE_URL, VLM_API_KEY, VLM_MODEL)
else:
    print("VLM skipped: paste VLM_API_BASE_URL and VLM_API_KEY above to enable section 3.")


VLM endpoint configured: https://api.gpugeek.com/v1  model=gpt-5.5


## 3. VLM feature smoke

One feature, two images. Inputs are the per-channel PNGs under `slices/`.


In [5]:
import json

vlm_feature = "vlm_nuclear_cap_presence"
spec = json.loads((P.FEATURE_LIBRARY / "vlm" / vlm_feature / "feature.json").read_text(encoding="utf-8"))
print("name       :", spec.get("name"))
print("category   :", spec.get("category"))
print("description:\n", spec.get("description"))

if not feat.vlm_api_ready():
    print("\nNo VLM credentials — live call skipped.")
    vlm_df = None
else:
    vlm_df = feat.run_vlm_feature(vlm_feature, P.DATASET_DIR, ids[:2])
    vlm_df.to_csv(feat.OUTPUT_SMOKE / f"smoke_vlm_{vlm_feature}.csv", index=False)
    display(vlm_df[["sample_id", vlm_feature, "error"]])


name       : vlm_nuclear_cap_presence
category   : morphology
description:
 Detects the presence of 'nuclear caps' (asymmetric clustering of organelles or nuclear envelope deformations), a phenotype linked to oxidative stress and specific drug mechanisms.
  [vlm_nuclear_cap_presence] 1/2 -> 27.4 
  [vlm_nuclear_cap_presence] 2/2 -> 21.7 
  vlm_nuclear_cap_presence: 2 samples in 37.32s (18.7s/sample)


,sample_id,vlm_nuclear_cap_presence,error
0,10064_alsterpaullone,27.4,None
1,10065_alsterpaullone,21.7,None


## 4. Full-dataset wall time

Smoke timings on this machine, extrapolated to 3,552 images:

| Workload | Approx. serial time |
|----------|---------------------|
| 439 code features | ~29 h (~2 h with 16 processes) |
| 28 VLM features, one call each | hundreds of hours |
| VLM batched (1 call / image) | ~14 h, depends on your API |

CLI:

```bash
python code/extract_morphagent_features.py --smoke
python code/extract_morphagent_features.py --smoke --api-base https://YOUR_HOST/v1 --api-key YOUR_KEY --api-model YOUR_MODEL
```
